<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Exorust/TorchLeet/blob/main/torch/medium/cnn-scratch/CNN_scratch.ipynb)

Check your work: `!pip install torchleet` then `from torchleet import check; check("cnn-scratch", ...)`


# Problem: Implement a CNN for CIFAR-10 (With Custom Layers)

### Problem Statement
You are tasked with implementing a **Convolutional Neural Network (CNN)** for image classification on the **CIFAR-10** dataset using PyTorch. However, instead of using PyTorch's built-in `nn.Conv2d` and `nn.MaxPool2d`, you must implement these layers **from scratch** using `nn.Module`. Your model will include convolutional layers for feature extraction, pooling layers for downsampling, and fully connected layers for classification.

### Requirements
1. **Implement Custom Layers**:
   - Create a custom `Conv2dCustom` class that mimics the behavior of `nn.Conv2d`.
   - Create a custom `MaxPool2dCustom` class that mimics the behavior of `nn.MaxPool2d`.

2. **Define the CNN Model**:
   - Use `Conv2dCustom` for convolutional layers.
   - Use `MaxPool2dCustom` for pooling layers.
   - Use standard `nn.Linear` for fully connected layers.
   - The model should process input images of shape `(3, 32, 32)` as in the CIFAR-10 dataset.

### Constraints
- You must not use `nn.Conv2d` or `nn.MaxPool2d`. Use your own custom implementations.
- The CNN should include multiple convolutional and pooling layers, followed by fully connected layers.
- Ensure the model outputs class predictions for **10 classes**, as required by CIFAR-10.

<details>
  <summary>💡 Hint</summary>
  Define `Conv2dCustom` and `MaxPool2dCustom` as subclasses of `nn.Module`. Use nested loops and tensor slicing to perform the operations.  
  In `CNNModel.__init__`, use these custom layers to build the architecture.  
  Implement the forward pass to pass inputs through convolution, activation, pooling, flattening, and fully connected layers.
</details>


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F

In [2]:
# Load CIFAR-10 dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)

test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

100%|██████████| 170M/170M [25:56<00:00, 110kB/s]


In [4]:
class Conv2dCustom(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, dilation=1):
        super(Conv2dCustom, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size if isinstance(kernel_size, tuple) else (kernel_size, kernel_size)
        self.stride = stride
        self.padding = padding
        self.dilation = dilation
        self.weights = nn.Parameter(torch.randn(out_channels, in_channels, *self.kernel_size))
        self.bias = nn.Parameter(torch.randn(out_channels))

    def forward(self, x):
        B, C, H, W = x.shape

        KH, KW = self.kernel_size
        SH = SW = self.stride
        PH = PW = self.padding
        DH = DW = self.dilation

        KH_eff = DH*(KH-1)+1
        KW_eff = DW*(KW-1)+1

        x_padded = F.pad(x, (PW, PW, PH, PH))

        H_out = (H + 2*PH - KH_eff)//SH + 1
        W_out = (W + 2*PW - KW_eff)//SW + 1

        output = torch.zeros((B, self.out_channels, H_out, W_out), device=x.device)

        for b in range(B):
          for oc in range(self.out_channels):
            for j in range(H_out):
                for k in range(W_out):
                  region = x_padded[B,:,j*SH:j*SH+KH_eff,k*SW:k*SW+KW_eff]
                  output[B,oc,j,k] = torch.sum(region*self.weights[oc]) + self.bias[oc]

        return output

class MaxPool2dCustom(nn.Module):
    def __init__(self, kernel_size, stride=None, dilation=0):
        super(MaxPool2dCustom, self).__init__()
        self.kernel_size = kernel_size if isinstance(kernel_size, tuple) else (kernel_size, kernel_size)
        self.stride = stride if stride is not None else kernel_size
        self.dilation = dilation

    def forward(self, x):
        B, C, H, W = x.shape

        KH, KW = self.kernel_size
        SH = SW = self.stride
        DH = DW = self.dilation
        H_out = (H-KH)//SH + 1
        W_out = (W-KW)//SW + 1

        KH_eff = DH*(KH-1)+1
        KW_eff = DW*(KW-1)+1

        output = torch.zeros((B, C, H_out, W_out),device=x.device)

        for b in range(B):
          for c in range(C):
            for i in range(H_out):
              for j in range(W_out):
                region = x[b,c,i*SH:i*SH+KH_eff,j*SW:j*SW+KW_eff]
                output[b,c,i,j] = torch.max(region)

        return output


In [5]:
# Define the CNN Model
class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)  # Output: 32x32x32
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)  # Output: 64x32x32
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)  # Output: 64x16x16
        self.fc1 = nn.Linear(64 * 16 * 16, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)  # Flatten
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
# Initialize the model, loss function, and optimizer
model = CNNModel()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
epochs = 10
for epoch in range(epochs):
    for images, labels in train_loader:
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [1/10], Loss: 0.9105
Epoch [2/10], Loss: 0.9879
Epoch [3/10], Loss: 0.5569
Epoch [4/10], Loss: 0.8733
Epoch [5/10], Loss: 0.5920


In [ ]:
# Evaluate on the test set
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")